In [ ]:
import asyncio
from ib_insync import *
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import random
from pymongo import MongoClient
from bson import ObjectId
import warnings

# Suppress warnings
warnings.filterwarnings("ignore")
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=UserWarning)

# Connect to MongoDB
client = MongoClient('mongodb://localhost:27017/')
db = client['trading_db']
trades_collection = db['trades_v2']
exclude_tickers_collection = db['excluded_tickers_v2']

# Connect to IB
util.startLoop()
ib = IB()
ib.connect('127.0.0.1', 7497, clientId=random.randint(1, 1000))

#tickers = ['AAPL', 'TSLA', 'AMZN', 'GOOG', 'COIN', 'NVDA', 'BA', 'TSM', 'MSFT', 'META', 'NFLX', 'AMD']
tickers = [ 'WMT', 'MU', 'SAVA', 'SLDB', 'SLV', 'NEM', 'CTXR', 'ONON', 'IONQ', 'MARA', 'MRVL', 'T', 'CCL', 'XYZ', 'PG', 'ONDS', 'NFLX', 'V', 'ADBE', 'MS', 'CXW', 'BA', 'BABA', 'DAL', 'JPM', 'C', 'AMZN', 'UAL', 'BRK.B', 'IBM', 'MSFT', 'KO', 'MSTR', 'COIN', 'GLD', 'META', 'AAL', 'OSCR', 'QSI', 'SPY', 'NVO', 'DIS', 'AAPL', 'BMNR', 'GRAB', 'RBLX', 'AFRM', 'NVDA', 'FUBO', 'PYPL', 'GOOG', 'BTC', 'RGTI', 'GPRO', 'OKLO', 'AMD', 'XPEV', 'SHEL', 'TSM', 'SNOW', 'NET', 'SES', 'TSLA', 'BP', 'UBER', 'INTC', 'MRNA', 'IREN', 'ORCL', 'HIMS', 'NBIS']
contracts = [Stock(ticker, 'SMART', 'USD') for ticker in tickers]
ib.qualifyContracts(*contracts)


# ══════════════════════════════════════════════════════════════════════════════
# EMA ALIGNMENT OSCILLATOR CALCULATION
# ══════════════════════════════════════════════════════════════════════════════

def calculate_ema(data, period):
    """Calculate Exponential Moving Average"""
    return data['close'].ewm(span=period, adjust=False).mean()


def calculate_atr(df, period=14):
    """Calculate Average True Range"""
    high_low = df['high'] - df['low']
    high_close = np.abs(df['high'] - df['close'].shift())
    low_close = np.abs(df['low'] - df['close'].shift())
    ranges = pd.concat([high_low, high_close, low_close], axis=1)
    true_range = np.max(ranges, axis=1)
    atr = true_range.rolling(period).mean()
    df['ATR'] = atr
    return df


def calculate_ema_200(df):
    """Calculate 200 EMA for trend filter"""
    df['ema_200'] = calculate_ema(df, 200)
    return df


def calculate_ema_alignment_oscillator(df, 
                                        fast_period=9, 
                                        mid_period=14, 
                                        slow_period=21, 
                                        trend_period=50,
                                        slope_threshold=2.0,
                                        spacing_min=0.1):
    """
    Calculate EMA Alignment Strength Oscillator (0-100 scale)
    Returns DataFrame with 'trend_score' column
    """
    # Calculate EMAs
    df['ema_fast'] = calculate_ema(df, fast_period)
    df['ema_mid'] = calculate_ema(df, mid_period)
    df['ema_slow'] = calculate_ema(df, slow_period)
    df['ema_trend'] = calculate_ema(df, trend_period)
    
    # Calculate ATR for normalization
    df = calculate_atr(df, 14)
    
    # Calculate slopes in degrees (normalized by ATR)
    def slope_degrees(ema_series, atr_series):
        slope = (ema_series - ema_series.shift(1)) / (atr_series + 0.0001) * 100
        return np.degrees(np.arctan(slope))
    
    df['fast_angle'] = slope_degrees(df['ema_fast'], df['ATR'])
    df['mid_angle'] = slope_degrees(df['ema_mid'], df['ATR'])
    df['slow_angle'] = slope_degrees(df['ema_slow'], df['ATR'])
    
    # Alignment conditions
    df['bullish_stack'] = (df['ema_fast'] > df['ema_mid']) & \
                          (df['ema_mid'] > df['ema_slow']) & \
                          (df['ema_slow'] > df['ema_trend'])
    
    df['bearish_stack'] = (df['ema_fast'] < df['ema_mid']) & \
                          (df['ema_mid'] < df['ema_slow']) & \
                          (df['ema_slow'] < df['ema_trend'])
    
    # Price position
    df['price_above_fast'] = df['close'] > df['ema_fast']
    df['price_below_fast'] = df['close'] < df['ema_fast']
    
    # Spacing analysis
    df['spacing_fast_mid'] = np.abs(df['ema_fast'] - df['ema_mid']) / df['ema_mid'] * 100
    df['spacing_mid_slow'] = np.abs(df['ema_mid'] - df['ema_slow']) / df['ema_slow'] * 100
    df['avg_spacing'] = (df['spacing_fast_mid'] + df['spacing_mid_slow']) / 2
    
    # Volume confirmation (vs 20-period average)
    df['volume_sma'] = df['volume'].rolling(20).mean()
    df['volume_confirmation'] = df['volume'] > df['volume_sma']
    
    # Calculate Trend Score
    df['trend_score'] = 0.0
    
    for i in range(1, len(df)):
        # Bullish alignment
        if df['bullish_stack'].iloc[i] and df['price_above_fast'].iloc[i]:
            base_score = 40.0
            
            # Slope bonus (0-30)
            avg_slope = (df['fast_angle'].iloc[i] + df['mid_angle'].iloc[i] + df['slow_angle'].iloc[i]) / 3
            slope_bonus = min(avg_slope * 3, 30) if avg_slope > slope_threshold else 0
            
            # Spacing bonus (0-20)
            spacing_bonus = min(df['avg_spacing'].iloc[i] * 10, 20) if df['avg_spacing'].iloc[i] > spacing_min else 0
            
            # Volume bonus (0-10)
            vol_bonus = 10 if df['volume_confirmation'].iloc[i] else 0
            
            new_score = base_score + slope_bonus + spacing_bonus + vol_bonus
            df.loc[df.index[i], 'trend_score'] = (new_score + df['trend_score'].iloc[i-1]) / 2  # Smooth
        
        # Bearish alignment
        elif df['bearish_stack'].iloc[i] and df['price_below_fast'].iloc[i]:
            base_score = -40.0
            
            avg_slope = (abs(df['fast_angle'].iloc[i]) + abs(df['mid_angle'].iloc[i]) + abs(df['slow_angle'].iloc[i])) / 3
            slope_bonus = -min(avg_slope * 3, 30) if avg_slope > slope_threshold else 0
            spacing_bonus = -min(df['avg_spacing'].iloc[i] * 10, 20) if df['avg_spacing'].iloc[i] > spacing_min else 0
            vol_bonus = -10 if df['volume_confirmation'].iloc[i] else 0
            
            new_score = base_score + slope_bonus + spacing_bonus + vol_bonus
            df.loc[df.index[i], 'trend_score'] = (new_score + df['trend_score'].iloc[i-1]) / 2
        
        else:
            # Decay when not aligned
            df.loc[df.index[i], 'trend_score'] = df['trend_score'].iloc[i-1] * 0.9
    
    # Volume filter
    df.loc[~df['volume_confirmation'], 'trend_score'] *= 0.8
    
    # Clamp between -100 and 100
    df['trend_score'] = df['trend_score'].clip(-100, 100)
    
    # Generate signals
    df['strong_bull_signal'] = (df['trend_score'] >= 70) & (df['trend_score'] > df['trend_score'].shift(1))
    df['strong_bear_signal'] = (df['trend_score'] <= -70) & (df['trend_score'] < df['trend_score'].shift(1))
    
    return df


# ══════════════════════════════════════════════════════════════════════════════
# TRADE DOCUMENT FUNCTIONS
# ══════════════════════════════════════════════════════════════════════════════

def create_trade_document(symbol, direction, entry_price, quantity, trend_score_at_entry):
    return {
        'instrument': symbol,
        'direction': direction,
        'entry_price': entry_price,
        'highest_price': entry_price,
        'quantity': quantity,
        'entry_timestamp': datetime.now(),
        'entry_trend_score': trend_score_at_entry,  # Store trend score at entry
        'order_id': str(ObjectId()),
        'exit_price': None,
        'exit_timestamp': None,
        'reason': None,
        'current_pnl': 0,
        'realized_pnl': 0,
        'status': 'open',
        'created_at': datetime.now(),
        'updated_at': datetime.now()
    }


def update_trade_document(trade_id, updates):
    updates['updated_at'] = datetime.now()
    trades_collection.update_one(
        {'_id': ObjectId(trade_id)},
        {'$set': updates}
    )


# ══════════════════════════════════════════════════════════════════════════════
# MAIN TRADING LOOP
# ══════════════════════════════════════════════════════════════════════════════

async def check_and_trade():
    # Position management parameters
    base_trailing_amount = 1.0
    min_trailing_pct = 0.005
    atr_multiplier = 1.5
    
    # Stop loss parameters
    stop_loss_1 = 0.01  # 1% stop loss
    stop_loss_2 = 0.02  # 2% stop loss
    
    # Take profit parameters
    take_profit_pct = 0.01  # 1% profit
    take_profit_dollar = 2.0  # $2 profit per share

    while True:
        # Fetch current positions
        positions = ib.positions()
        position_sizes = {pos.contract.symbol: pos.position for pos in positions}

        for contract in contracts:
            symbol = contract.symbol
            print(f"\n{'='*60}")
            print(f"Evaluating {symbol} at {datetime.now().strftime('%H:%M:%S')}")
            print(f"{'='*60}")

            # Fetch historical data (5-minute bars)
            bars = ib.reqHistoricalData(
                contract,
                endDateTime='',
                durationStr='2 D',
                barSizeSetting='5 mins',
                whatToShow='TRADES',
                useRTH=False,
                formatDate=1
            )

            if not bars:
                print(f"No historical data for {symbol}")
                continue

            # Convert to DataFrame
            df = util.df(bars)
            
            # Rename columns to match our functions (lowercase)
            df.columns = [col.lower() for col in df.columns]
            
            # Calculate EMA Alignment Oscillator
            df = calculate_ema_alignment_oscillator(df)
            df = calculate_ema_200(df)  # Also calculate 200 EMA
            
            current_row = df.iloc[-1]
            current_price = current_row['close']
            current_trend_score = current_row['trend_score']
            current_atr = current_row.get('ATR', base_trailing_amount)
            
            print(f"Current Price: ${current_price:.2f}")
            print(f"Trend Score: {current_trend_score:.1f}/100")
            print(f"EMA 200: ${current_row['ema_200']:.2f}")
            print(f"Close > EMA200: {current_price > current_row['ema_200']}")
            print(f"Strong Bull Signal: {current_row['strong_bull_signal']}")
            
            # Check for open trades
            open_trade = trades_collection.find_one({
                'instrument': symbol,
                'status': 'open'
            })

            # ═════════════════════════════════════════════════════════════════
            # POSITION MANAGEMENT (SELL LOGIC)
            # ═════════════════════════════════════════════════════════════════
            
            if open_trade:
                entry_price = open_trade['entry_price']
                highest_price = open_trade.get('highest_price', entry_price)
                quantity = open_trade['quantity']
                time_in_trade = (datetime.now() - open_trade['entry_timestamp']).total_seconds() / 3600
                
                # Update highest price if current price is higher
                if current_price > highest_price:
                    highest_price = current_price
                    update_trade_document(open_trade['_id'], {
                        'highest_price': highest_price
                    })

                # Calculate P&L
                profit_pct = (current_price - entry_price) / entry_price
                profit_loss_per_share = current_price - entry_price
                current_pnl = profit_loss_per_share * quantity
                
                update_trade_document(open_trade['_id'], {
                    'current_pnl': current_pnl
                })
                
                print(f"Open Position - Entry: ${entry_price:.2f}, P&L: {profit_pct:.2%}, Trend: {current_trend_score:.1f}")
                
                # ═════════════════════════════════════════════════════════════
                # STOP LOSS LOGIC (Updated with Trend Score Check)
                # ═════════════════════════════════════════════════════════════
                
                should_exit_on_loss = False
                exit_reason = None
                
                if profit_pct <= -stop_loss_1:  # At 1% loss
                    if current_trend_score < 69:  # Trend score dropped below 69 (not strong anymore)
                        should_exit_on_loss = True
                        exit_reason = 'stop_loss_1pct_trend_weak'
                        print(f"🚨 EXIT TRIGGER: 1% loss + Trend score {current_trend_score:.1f} < 69")
                    elif profit_pct <= -stop_loss_2:  # At 2% loss (hard stop)
                        should_exit_on_loss = True
                        exit_reason = 'stop_loss_2pct_max'
                        print(f"🚨 EXIT TRIGGER: 2% loss maximum reached")
                    else:
                        print(f"Holding at {profit_pct:.2%} loss - Trend still strong ({current_trend_score:.1f})")
                
                # Execute stop loss exit
                if should_exit_on_loss and quantity > 0:
                    order = MarketOrder('SELL', quantity)
                    trade = ib.placeOrder(contract, order)

                    realized_pnl = (current_price - entry_price) * quantity

                    update_trade_document(open_trade['_id'], {
                        'exit_price': current_price,
                        'exit_timestamp': datetime.now(),
                        'reason': exit_reason,
                        'realized_pnl': realized_pnl,
                        'exit_trend_score': current_trend_score,
                        'status': 'closed'
                    })

                    print(f"✅ STOP LOSS executed for {symbol} at ${current_price:.2f} | P&L: ${realized_pnl:.2f}")
                    continue
                
                # ═════════════════════════════════════════════════════════════
                # TAKE PROFIT LOGIC (Unchanged)
                # ═════════════════════════════════════════════════════════════
                
                if profit_pct >= take_profit_pct or profit_loss_per_share >= take_profit_dollar:
                    order = MarketOrder('SELL', quantity)
                    trade = ib.placeOrder(contract, order)

                    realized_pnl = (current_price - entry_price) * quantity

                    update_trade_document(open_trade['_id'], {
                        'exit_price': current_price,
                        'exit_timestamp': datetime.now(),
                        'reason': f'take_profit_{profit_pct:.2%}_or_${profit_loss_per_share:.2f}',
                        'realized_pnl': realized_pnl,
                        'exit_trend_score': current_trend_score,
                        'status': 'closed'
                    })

                    print(f"✅ TAKE PROFIT executed for {symbol} at ${current_price:.2f} | P&L: ${realized_pnl:.2f}")
                    continue
                
                # ═════════════════════════════════════════════════════════════
                # TRAILING STOP LOGIC (Unchanged)
                # ═════════════════════════════════════════════════════════════
                
                if profit_pct > 0:
                    atr_based_stop = highest_price - (current_atr * atr_multiplier)
                    percentage_based_stop = highest_price * (1 - min_trailing_pct)
                    fixed_based_stop = highest_price - base_trailing_amount
                    
                    time_factor = min(1.0, time_in_trade / 24)
                    trailing_stop_price = atr_based_stop * (1 - 0.25 * time_factor)
                    trailing_stop_price = max(trailing_stop_price, percentage_based_stop, fixed_based_stop)
                    
                    # Check for trailing stop hit
                    if current_price <= trailing_stop_price and quantity > 0:
                        order = MarketOrder('SELL', quantity)
                        trade = ib.placeOrder(contract, order)

                        realized_pnl = (current_price - entry_price) * quantity

                        update_trade_document(open_trade['_id'], {
                            'exit_price': current_price,
                            'exit_timestamp': datetime.now(),
                            'reason': 'trailing_stop_hit',
                            'realized_pnl': realized_pnl,
                            'exit_trend_score': current_trend_score,
                            'status': 'closed'
                        })

                        print(f"✅ TRAILING STOP executed for {symbol} at ${current_price:.2f} | P&L: ${realized_pnl:.2f}")
                        continue

            # ═════════════════════════════════════════════════════════════════
            # ENTRY MANAGEMENT (BUY LOGIC - Updated)
            # ═════════════════════════════════════════════════════════════════
            
            else:
                # Check if already traded today
                current_date = datetime.now().date()
                exclude_entry = exclude_tickers_collection.find_one({
                    'ticker': symbol,
                    'date': current_date.isoformat()
                })

                if exclude_entry:
                    print(f"Already traded {symbol} today. Skipping.")
                    continue

                # ═════════════════════════════════════════════════════════════
                # NEW ENTRY CONDITIONS (EMA Alignment Oscillator + EMA200)
                # ═════════════════════════════════════════════════════════════
                
                # Condition 1: Strong bull signal (trend score crossed above 70)
                # Condition 2: Close > EMA 200 (long-term trend filter)
                
                if current_row['strong_bull_signal'] and current_price > current_row['ema_200']:
                    
                    order_size = 20  # Adjust position size as needed
                    order = MarketOrder('BUY', order_size)
                    trade = ib.placeOrder(contract, order)

                    # Create trade document with trend score at entry
                    trade_doc = create_trade_document(
                        symbol=symbol,
                        direction='long',
                        entry_price=current_price,
                        quantity=order_size,
                        trend_score_at_entry=current_trend_score
                    )
                    
                    trades_collection.insert_one(trade_doc)

                    # Mark as traded today
                    exclude_tickers_collection.insert_one({
                        'ticker': symbol,
                        'date': current_date.isoformat()
                    })

                    print(f"🚀 NEW LONG POSITION: {symbol}")
                    print(f"   Entry: ${current_price:.2f}")
                    print(f"   Size: {order_size} shares")
                    print(f"   Trend Score: {current_trend_score:.1f}/100")
                    print(f"   EMA200: ${current_row['ema_200']:.2f} (Close above: ✓)")
                    
                else:
                    # Debug why no entry
                    if not current_row['strong_bull_signal']:
                        print(f"No entry: No strong bull signal (trend: {current_trend_score:.1f})")
                    if current_price <= current_row['ema_200']:
                        print(f"No entry: Close ${current_price:.2f} below EMA200 ${current_row['ema_200']:.2f}")

            # Brief pause between tickers
            await asyncio.sleep(0.5)

        # Wait 5 minutes before next scan
        print(f"\n{'='*60}")
        print(f"Scan complete. Waiting 5 minutes...")
        print(f"{'='*60}")
        await asyncio.sleep(300)


# Run the trading loop
try:
    ib.run(check_and_trade())
except Exception as e:
    print(f"An error occurred: {e}")
    import traceback
    traceback.print_exc()
finally:
    ib.disconnect()
    print("Disconnected from IB.")

Error 200, reqId 5: No security definition has been found for the request, contract: Stock(symbol='SAVA', exchange='SMART', currency='USD')
Error 200, reqId 31: No security definition has been found for the request, contract: Stock(symbol='BRK.B', exchange='SMART', currency='USD')
Unknown contract: Stock(symbol='SAVA', exchange='SMART', currency='USD')
Unknown contract: Stock(symbol='BRK.B', exchange='SMART', currency='USD')



Evaluating WMT at 08:34:35
Current Price: $122.34
Trend Score: -7.0/100
EMA 200: $122.28
Close > EMA200: True
Strong Bull Signal: False
No entry: No strong bull signal (trend: -7.0)

Evaluating MU at 08:34:36
Current Price: $387.70
Trend Score: -82.1/100
EMA 200: $398.95
Close > EMA200: False
Strong Bull Signal: False
No entry: No strong bull signal (trend: -82.1)
No entry: Close $387.70 below EMA200 $398.95


Error 200, reqId 76: No security definition has been found for the request, contract: Stock(symbol='SAVA', exchange='SMART', currency='USD')



Evaluating SAVA at 08:34:36
No historical data for SAVA

Evaluating SLDB at 08:34:36
Current Price: $6.81
Trend Score: 56.1/100
EMA 200: $6.81
Close > EMA200: True
Strong Bull Signal: False
No entry: No strong bull signal (trend: 56.1)

Evaluating SLV at 08:34:37
Current Price: $65.89
Trend Score: 65.6/100
EMA 200: $64.62
Close > EMA200: True
Strong Bull Signal: False
No entry: No strong bull signal (trend: 65.6)

Evaluating NEM at 08:34:38
Current Price: $103.20
Trend Score: 45.9/100
EMA 200: $101.38
Close > EMA200: True
Strong Bull Signal: False
No entry: No strong bull signal (trend: 45.9)

Evaluating CTXR at 08:34:39
Current Price: $0.73
Trend Score: 56.3/100
EMA 200: $0.72
Close > EMA200: True
Strong Bull Signal: False
No entry: No strong bull signal (trend: 56.3)

Evaluating ONON at 08:34:40
Current Price: $37.41
Trend Score: -57.7/100
EMA 200: $39.35
Close > EMA200: False
Strong Bull Signal: False
No entry: No strong bull signal (trend: -57.7)
No entry: Close $37.41 below EMA20

Error 10349, reqId 89: Order TIF was set to GTC based on order preset.
Canceled order: Trade(contract=Stock(conId=11054, symbol='PG', exchange='SMART', primaryExchange='NYSE', currency='USD', localSymbol='PG', tradingClass='PG'), order=MarketOrder(orderId=89, clientId=8, action='BUY', totalQuantity=20), orderStatus=OrderStatus(orderId=89, status='Cancelled', filled=0.0, remaining=0.0, avgFillPrice=0.0, permId=0, parentId=0, lastFillPrice=0.0, clientId=0, whyHeld='', mktCapPrice=0.0), fills=[], log=[TradeLogEntry(time=datetime.datetime(2026, 3, 25, 12, 34, 47, 409902, tzinfo=datetime.timezone.utc), status='PendingSubmit', message='', errorCode=0), TradeLogEntry(time=datetime.datetime(2026, 3, 25, 12, 34, 47, 448236, tzinfo=datetime.timezone.utc), status='Cancelled', message='Error 10349, reqId 89: Order TIF was set to GTC based on order preset.', errorCode=10349)], advancedError='')


...

Current Price: $92.95
Trend Score: -82.4/100
EMA 200: $94.95
Close > EMA200: False
Strong Bull Signal: False
Already traded UAL today. Skipping.

Evaluating BRK.B at 16:01:17
No historical data for BRK.B

Evaluating IBM at 16:01:17
Current Price: $241.39
Trend Score: -53.5/100
EMA 200: $242.39
Close > EMA200: False
Strong Bull Signal: False
Already traded IBM today. Skipping.

Evaluating MSFT at 16:01:18
Current Price: $371.17
Trend Score: -25.9/100
EMA 200: $373.72
Close > EMA200: False
Strong Bull Signal: False
No entry: No strong bull signal (trend: -25.9)
No entry: Close $371.17 below EMA200 $373.72

Evaluating KO at 16:01:19
Current Price: $75.25
Trend Score: 6.4/100
EMA 200: $75.14
Close > EMA200: True
Strong Bull Signal: False
No entry: No strong bull signal (trend: 6.4)

Evaluating MSTR at 16:01:19
Current Price: $138.86
Trend Score: -24.4/100
EMA 200: $139.27
Close > EMA200: False
Strong Bull Signal: False
Already traded MSTR today. Skipping.

Evaluating COIN at 16:01:20
Curre

Error 200, reqId 5766: No security definition has been found for the request, contract: Stock(symbol='SAVA', exchange='SMART', currency='USD')



Evaluating SAVA at 16:06:43
No historical data for SAVA

Evaluating SLDB at 16:06:43
Current Price: $7.23
Trend Score: 2.8/100
EMA 200: $7.03
Close > EMA200: True
Strong Bull Signal: False
Already traded SLDB today. Skipping.

Evaluating SLV at 16:06:44
Current Price: $64.97
Trend Score: -53.1/100
EMA 200: $65.18
Close > EMA200: False
Strong Bull Signal: False
No entry: No strong bull signal (trend: -53.1)
No entry: Close $64.97 below EMA200 $65.18

Evaluating NEM at 16:06:45
Current Price: $101.76
Trend Score: -50.6/100
EMA 200: $101.91
Close > EMA200: False
Strong Bull Signal: False
No entry: No strong bull signal (trend: -50.6)
No entry: Close $101.76 below EMA200 $101.91

Evaluating CTXR at 16:06:45
Current Price: $0.71
Trend Score: -48.3/100
EMA 200: $0.73
Close > EMA200: False
Strong Bull Signal: False
Already traded CTXR today. Skipping.

Evaluating ONON at 16:06:46
Current Price: $35.26
Trend Score: -43.9/100
EMA 200: $37.12
Close > EMA200: False
Strong Bull Signal: False
No e

Error 200, reqId 5792: No security definition has been found for the request, contract: Stock(symbol='BRK.B', exchange='SMART', currency='USD')


Current Price: $92.95
Trend Score: -61.9/100
EMA 200: $94.93
Close > EMA200: False
Strong Bull Signal: False
Already traded UAL today. Skipping.

Evaluating BRK.B at 16:06:58
No historical data for BRK.B

Evaluating IBM at 16:06:58
Current Price: $241.10
Trend Score: -55.9/100
EMA 200: $242.38
Close > EMA200: False
Strong Bull Signal: False
Already traded IBM today. Skipping.

Evaluating MSFT at 16:06:58
Current Price: $371.15
Trend Score: -18.7/100
EMA 200: $373.70
Close > EMA200: False
Strong Bull Signal: False
No entry: No strong bull signal (trend: -18.7)
No entry: Close $371.15 below EMA200 $373.70

Evaluating KO at 16:06:59
Current Price: $75.26
Trend Score: 4.6/100
EMA 200: $75.15
Close > EMA200: True
Strong Bull Signal: False
No entry: No strong bull signal (trend: 4.6)

Evaluating MSTR at 16:07:00
Current Price: $138.55
Trend Score: -17.5/100
EMA 200: $139.26
Close > EMA200: False
Strong Bull Signal: False
Already traded MSTR today. Skipping.

Evaluating COIN at 16:07:00
Curre

In [ ]:
import asyncio
from ib_insync import *
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import random
from pymongo import MongoClient
from bson import ObjectId
import warnings

# Suppress warnings
warnings.filterwarnings("ignore")
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=UserWarning)

# Connect to MongoDB
client = MongoClient('mongodb://localhost:27017/')
db = client['trading_db']
trades_collection = db['trades_v2']
exclude_tickers_collection = db['excluded_tickers_v2']

# Connect to IB
util.startLoop()
ib = IB()
ib.connect('127.0.0.1', 7497, clientId=random.randint(1, 1000))

#tickers = ['AAPL', 'TSLA', 'AMZN', 'GOOG', 'COIN', 'NVDA', 'BA', 'TSM', 'MSFT', 'META', 'NFLX', 'AMD']
tickers = [ 'WMT', 'MU', 'SAVA', 'SLDB', 'SLV', 'NEM', 'CTXR', 'ONON', 'IONQ', 'MARA', 'MRVL', 'T', 'CCL', 'XYZ', 'PG', 'ONDS', 'NFLX', 'V', 'ADBE', 'MS', 'CXW', 'BA', 'BABA', 'DAL', 'JPM', 'C', 'AMZN', 'UAL', 'BRK.B', 'IBM', 'MSFT', 'KO', 'MSTR', 'COIN', 'GLD', 'META', 'AAL', 'OSCR', 'QSI', 'SPY', 'NVO', 'DIS', 'AAPL', 'BMNR', 'GRAB', 'RBLX', 'AFRM', 'NVDA', 'FUBO', 'PYPL', 'GOOG', 'BTC', 'RGTI', 'GPRO', 'OKLO', 'AMD', 'XPEV', 'SHEL', 'TSM', 'SNOW', 'NET', 'SES', 'TSLA', 'BP', 'UBER', 'INTC', 'MRNA', 'IREN', 'ORCL', 'HIMS', 'NBIS']
contracts = [Stock(ticker, 'SMART', 'USD') for ticker in tickers]
ib.qualifyContracts(*contracts)


# ══════════════════════════════════════════════════════════════════════════════
# EMA ALIGNMENT OSCILLATOR CALCULATION
# ══════════════════════════════════════════════════════════════════════════════

def calculate_ema(data, period):
    """Calculate Exponential Moving Average"""
    return data['close'].ewm(span=period, adjust=False).mean()


def calculate_atr(df, period=14):
    """Calculate Average True Range"""
    high_low = df['high'] - df['low']
    high_close = np.abs(df['high'] - df['close'].shift())
    low_close = np.abs(df['low'] - df['close'].shift())
    ranges = pd.concat([high_low, high_close, low_close], axis=1)
    true_range = np.max(ranges, axis=1)
    atr = true_range.rolling(period).mean()
    df['ATR'] = atr
    return df


def calculate_ema_200(df):
    """Calculate 200 EMA for trend filter"""
    df['ema_200'] = calculate_ema(df, 200)
    return df


def calculate_session_vwap(df):
    """
    Calculate Session VWAP — resets each trading day.
    Uses today's bars only so VWAP is a true intraday measure.
    """
    df = df.copy()

    # Ensure date column is parsed
    df['date_only'] = pd.to_datetime(df['date']).dt.date
    today = df['date_only'].max()

    # Compute VWAP only for today's bars
    today_mask = df['date_only'] == today
    today_df = df.loc[today_mask].copy()

    today_df['vwap'] = (
        (today_df['close'] * today_df['volume']).cumsum() /
        today_df['volume'].cumsum()
    )

    df.loc[today_mask, 'vwap'] = today_df['vwap'].values

    # Forward-fill so iloc[-1] always has a valid VWAP value
    df['vwap'] = df['vwap'].ffill()

    return df


def calculate_ema_alignment_oscillator(df, 
                                        fast_period=8, 
                                        mid_period=13, 
                                        slow_period=14, 
                                        trend_period=21,
                                        slope_threshold=1.0,
                                        spacing_min=0.2):
    """
    Calculate EMA Alignment Strength Oscillator (0-100 scale)
    Returns DataFrame with 'trend_score' column
    """
    # Calculate EMAs
    df['ema_fast'] = calculate_ema(df, fast_period)
    df['ema_mid'] = calculate_ema(df, mid_period)
    df['ema_slow'] = calculate_ema(df, slow_period)
    df['ema_trend'] = calculate_ema(df, trend_period)
    
    # Calculate ATR for normalization
    df = calculate_atr(df, 14)
    
    # Calculate slopes in degrees (normalized by ATR)
    def slope_degrees(ema_series, atr_series):
        slope = (ema_series - ema_series.shift(1)) / (atr_series + 0.0001) * 100
        return np.degrees(np.arctan(slope))
    
    df['fast_angle'] = slope_degrees(df['ema_fast'], df['ATR'])
    df['mid_angle'] = slope_degrees(df['ema_mid'], df['ATR'])
    df['slow_angle'] = slope_degrees(df['ema_slow'], df['ATR'])
    
    # Alignment conditions
    df['bullish_stack'] = (df['ema_fast'] > df['ema_mid']) & \
                          (df['ema_mid'] > df['ema_slow']) & \
                          (df['ema_slow'] > df['ema_trend'])
    
    df['bearish_stack'] = (df['ema_fast'] < df['ema_mid']) & \
                          (df['ema_mid'] < df['ema_slow']) & \
                          (df['ema_slow'] < df['ema_trend'])
    
    # Price position
    df['price_above_fast'] = df['close'] > df['ema_fast']
    df['price_below_fast'] = df['close'] < df['ema_fast']
    
    # Spacing analysis
    df['spacing_fast_mid'] = np.abs(df['ema_fast'] - df['ema_mid']) / df['ema_mid'] * 100
    df['spacing_mid_slow'] = np.abs(df['ema_mid'] - df['ema_slow']) / df['ema_slow'] * 100
    df['avg_spacing'] = (df['spacing_fast_mid'] + df['spacing_mid_slow']) / 2
    
    # Volume confirmation (vs 20-period average)
    df['volume_sma'] = df['volume'].rolling(20).mean()
    df['volume_confirmation'] = df['volume'] > df['volume_sma']
    
    # Calculate Trend Score
    df['trend_score'] = 0.0
    
    for i in range(1, len(df)):
        # Bullish alignment
        if df['bullish_stack'].iloc[i] and df['price_above_fast'].iloc[i]:
            base_score = 40.0
            
            # Slope bonus (0-30)
            avg_slope = (df['fast_angle'].iloc[i] + df['mid_angle'].iloc[i] + df['slow_angle'].iloc[i]) / 3
            slope_bonus = min(avg_slope * 3, 30) if avg_slope > slope_threshold else 0
            
            # Spacing bonus (0-20)
            spacing_bonus = min(df['avg_spacing'].iloc[i] * 10, 20) if df['avg_spacing'].iloc[i] > spacing_min else 0
            
            # Volume bonus (0-10)
            vol_bonus = 10 if df['volume_confirmation'].iloc[i] else 0
            
            new_score = base_score + slope_bonus + spacing_bonus + vol_bonus
            df.loc[df.index[i], 'trend_score'] = (new_score + df['trend_score'].iloc[i-1]) / 2  # Smooth
        
        # Bearish alignment
        elif df['bearish_stack'].iloc[i] and df['price_below_fast'].iloc[i]:
            base_score = -40.0
            
            avg_slope = (abs(df['fast_angle'].iloc[i]) + abs(df['mid_angle'].iloc[i]) + abs(df['slow_angle'].iloc[i])) / 3
            slope_bonus = -min(avg_slope * 3, 30) if avg_slope > slope_threshold else 0
            spacing_bonus = -min(df['avg_spacing'].iloc[i] * 10, 20) if df['avg_spacing'].iloc[i] > spacing_min else 0
            vol_bonus = -10 if df['volume_confirmation'].iloc[i] else 0
            
            new_score = base_score + slope_bonus + spacing_bonus + vol_bonus
            df.loc[df.index[i], 'trend_score'] = (new_score + df['trend_score'].iloc[i-1]) / 2
        
        else:
            # Decay when not aligned
            df.loc[df.index[i], 'trend_score'] = df['trend_score'].iloc[i-1] * 0.9
    
    # Volume filter
    df.loc[~df['volume_confirmation'], 'trend_score'] *= 0.8
    
    # Clamp between -100 and 100
    df['trend_score'] = df['trend_score'].clip(-100, 100)
    
    # Generate signals
    df['strong_bull_signal'] = (df['trend_score'] >= 75) & (df['trend_score'] > df['trend_score'].shift(1))
    df['strong_bear_signal'] = (df['trend_score'] <= -75) & (df['trend_score'] < df['trend_score'].shift(1))
    
    return df


# ══════════════════════════════════════════════════════════════════════════════
# TRADE DOCUMENT FUNCTIONS
# ══════════════════════════════════════════════════════════════════════════════

def create_trade_document(symbol, direction, entry_price, quantity, trend_score_at_entry):
    return {
        'instrument': symbol,
        'direction': direction,
        'entry_price': entry_price,
        'highest_price': entry_price,
        'quantity': quantity,
        'entry_timestamp': datetime.now(),
        'entry_trend_score': trend_score_at_entry,  # Store trend score at entry
        'order_id': str(ObjectId()),
        'exit_price': None,
        'exit_timestamp': None,
        'reason': None,
        'current_pnl': 0,
        'realized_pnl': 0,
        'status': 'open',
        'created_at': datetime.now(),
        'updated_at': datetime.now()
    }


def update_trade_document(trade_id, updates):
    updates['updated_at'] = datetime.now()
    trades_collection.update_one(
        {'_id': ObjectId(trade_id)},
        {'$set': updates}
    )


# ══════════════════════════════════════════════════════════════════════════════
# MAIN TRADING LOOP
# ══════════════════════════════════════════════════════════════════════════════

async def check_and_trade():
    # Position management parameters
    base_trailing_amount = 1.0
    min_trailing_pct = 0.005
    atr_multiplier = 1.5
    
    # Stop loss parameters
    stop_loss_1 = 0.01  # 1% stop loss
    stop_loss_2 = 0.02  # 2% stop loss
    
    # Take profit parameters
    take_profit_pct = 0.01  # 1% profit
    take_profit_dollar = 2.0  # $2 profit per share

    while True:
        # Fetch current positions
        positions = ib.positions()
        position_sizes = {pos.contract.symbol: pos.position for pos in positions}

        for contract in contracts:
            symbol = contract.symbol
            print(f"\n{'='*60}")
            print(f"Evaluating {symbol} at {datetime.now().strftime('%H:%M:%S')}")
            print(f"{'='*60}")

            # Fetch historical data (5-minute bars)
            bars = ib.reqHistoricalData(
                contract,
                endDateTime='',
                durationStr='2 D',
                barSizeSetting='5 mins',
                whatToShow='TRADES',
                useRTH=False,
                formatDate=1
            )

            if not bars:
                print(f"No historical data for {symbol}")
                continue

            # Convert to DataFrame
            df = util.df(bars)
            
            # Rename columns to match our functions (lowercase)
            df.columns = [col.lower() for col in df.columns]
            
            # Calculate indicators
            df = calculate_ema_alignment_oscillator(df)
            df = calculate_ema_200(df)       # 200 EMA trend filter
            df = calculate_session_vwap(df)  # Session VWAP
            
            current_row = df.iloc[-1]
            current_price = current_row['close']
            current_trend_score = current_row['trend_score']
            current_atr = current_row.get('ATR', base_trailing_amount)
            current_vwap = current_row['vwap']
            
            print(f"Current Price:    ${current_price:.2f}")
            print(f"Trend Score:       {current_trend_score:.1f}/100")
            print(f"EMA 200:          ${current_row['ema_200']:.2f}")
            print(f"Session VWAP:     ${current_vwap:.2f}")
            print(f"Close > EMA200:    {current_price > current_row['ema_200']}")
            print(f"Close > VWAP:      {current_price > current_vwap}")
            print(f"Strong Bull Signal: {current_row['strong_bull_signal']}")
            
            # Check for open trades
            open_trade = trades_collection.find_one({
                'instrument': symbol,
                'status': 'open'
            })

            # ═════════════════════════════════════════════════════════════════
            # POSITION MANAGEMENT (SELL LOGIC)
            # ═════════════════════════════════════════════════════════════════
            
            if open_trade:
                entry_price = open_trade['entry_price']
                highest_price = open_trade.get('highest_price', entry_price)
                quantity = open_trade['quantity']
                time_in_trade = (datetime.now() - open_trade['entry_timestamp']).total_seconds() / 3600
                
                # Update highest price if current price is higher
                if current_price > highest_price:
                    highest_price = current_price
                    update_trade_document(open_trade['_id'], {
                        'highest_price': highest_price
                    })

                # Calculate P&L
                profit_pct = (current_price - entry_price) / entry_price
                profit_loss_per_share = current_price - entry_price
                current_pnl = profit_loss_per_share * quantity
                
                update_trade_document(open_trade['_id'], {
                    'current_pnl': current_pnl
                })
                
                print(f"Open Position - Entry: ${entry_price:.2f}, P&L: {profit_pct:.2%}, Trend: {current_trend_score:.1f}")
                
                # ═════════════════════════════════════════════════════════════
                # STOP LOSS LOGIC (with Trend Score Check)
                # ═════════════════════════════════════════════════════════════
                
                should_exit_on_loss = False
                exit_reason = None
                
                if profit_pct <= -stop_loss_1:  # At 1% loss
                    if current_trend_score < 69:  # Trend score dropped below 69 (not strong anymore)
                        should_exit_on_loss = True
                        exit_reason = 'stop_loss_1pct_trend_weak'
                        print(f"🚨 EXIT TRIGGER: 1% loss + Trend score {current_trend_score:.1f} < 69")
                    elif profit_pct <= -stop_loss_2:  # At 2% loss (hard stop)
                        should_exit_on_loss = True
                        exit_reason = 'stop_loss_2pct_max'
                        print(f"🚨 EXIT TRIGGER: 2% loss maximum reached")
                    else:
                        print(f"Holding at {profit_pct:.2%} loss - Trend still strong ({current_trend_score:.1f})")
                
                # Execute stop loss exit
                if should_exit_on_loss and quantity > 0:
                    order = MarketOrder('SELL', quantity)
                    trade = ib.placeOrder(contract, order)

                    realized_pnl = (current_price - entry_price) * quantity

                    update_trade_document(open_trade['_id'], {
                        'exit_price': current_price,
                        'exit_timestamp': datetime.now(),
                        'reason': exit_reason,
                        'realized_pnl': realized_pnl,
                        'exit_trend_score': current_trend_score,
                        'status': 'closed'
                    })

                    print(f"✅ STOP LOSS executed for {symbol} at ${current_price:.2f} | P&L: ${realized_pnl:.2f}")
                    continue
                
                # ═════════════════════════════════════════════════════════════
                # TAKE PROFIT LOGIC
                # ═════════════════════════════════════════════════════════════
                
                if profit_pct >= take_profit_pct or profit_loss_per_share >= take_profit_dollar:
                    order = MarketOrder('SELL', quantity)
                    trade = ib.placeOrder(contract, order)

                    realized_pnl = (current_price - entry_price) * quantity

                    update_trade_document(open_trade['_id'], {
                        'exit_price': current_price,
                        'exit_timestamp': datetime.now(),
                        'reason': f'take_profit_{profit_pct:.2%}_or_${profit_loss_per_share:.2f}',
                        'realized_pnl': realized_pnl,
                        'exit_trend_score': current_trend_score,
                        'status': 'closed'
                    })

                    print(f"✅ TAKE PROFIT executed for {symbol} at ${current_price:.2f} | P&L: ${realized_pnl:.2f}")
                    continue
                
                # ═════════════════════════════════════════════════════════════
                # TRAILING STOP LOGIC
                # ═════════════════════════════════════════════════════════════
                
                if profit_pct > 0:
                    atr_based_stop = highest_price - (current_atr * atr_multiplier)
                    percentage_based_stop = highest_price * (1 - min_trailing_pct)
                    fixed_based_stop = highest_price - base_trailing_amount
                    
                    time_factor = min(1.0, time_in_trade / 24)
                    trailing_stop_price = atr_based_stop * (1 - 0.25 * time_factor)
                    trailing_stop_price = max(trailing_stop_price, percentage_based_stop, fixed_based_stop)
                    
                    # Check for trailing stop hit
                    if current_price <= trailing_stop_price and quantity > 0:
                        order = MarketOrder('SELL', quantity)
                        trade = ib.placeOrder(contract, order)

                        realized_pnl = (current_price - entry_price) * quantity

                        update_trade_document(open_trade['_id'], {
                            'exit_price': current_price,
                            'exit_timestamp': datetime.now(),
                            'reason': 'trailing_stop_hit',
                            'realized_pnl': realized_pnl,
                            'exit_trend_score': current_trend_score,
                            'status': 'closed'
                        })

                        print(f"✅ TRAILING STOP executed for {symbol} at ${current_price:.2f} | P&L: ${realized_pnl:.2f}")
                        continue

            # ═════════════════════════════════════════════════════════════════
            # ENTRY MANAGEMENT (BUY LOGIC)
            # ═════════════════════════════════════════════════════════════════
            
            else:
                # Check if already traded today
                current_date = datetime.now().date()
                exclude_entry = exclude_tickers_collection.find_one({
                    'ticker': symbol,
                    'date': current_date.isoformat()
                })

                if exclude_entry:
                    print(f"Already traded {symbol} today. Skipping.")
                    continue

                # ═════════════════════════════════════════════════════════════
                # ENTRY CONDITIONS:
                #   1. Strong bull signal  (trend score crossed above 70)
                #   2. Close > EMA 200     (long-term trend filter)
                #   3. Close > Session VWAP (intraday momentum filter)  ← NEW
                # ═════════════════════════════════════════════════════════════
                
                above_ema200 = current_price > current_row['ema_200']
                above_vwap   = current_price > current_vwap

                if current_row['strong_bull_signal'] and above_ema200 and above_vwap:
                    
                    order_size = 20  # Adjust position size as needed
                    order = MarketOrder('BUY', order_size)
                    trade = ib.placeOrder(contract, order)

                    # Create trade document with trend score at entry
                    trade_doc = create_trade_document(
                        symbol=symbol,
                        direction='long',
                        entry_price=current_price,
                        quantity=order_size,
                        trend_score_at_entry=current_trend_score
                    )
                    
                    trades_collection.insert_one(trade_doc)

                    # Mark as traded today
                    exclude_tickers_collection.insert_one({
                        'ticker': symbol,
                        'date': current_date.isoformat()
                    })

                    print(f"🚀 NEW LONG POSITION: {symbol}")
                    print(f"   Entry:       ${current_price:.2f}")
                    print(f"   Size:         {order_size} shares")
                    print(f"   Trend Score:  {current_trend_score:.1f}/100")
                    print(f"   EMA200:      ${current_row['ema_200']:.2f} (Close above: ✓)")
                    print(f"   VWAP:        ${current_vwap:.2f} (Close above: ✓)")
                    
                else:
                    # Debug why no entry
                    if not current_row['strong_bull_signal']:
                        print(f"No entry: No strong bull signal (trend: {current_trend_score:.1f})")
                    if not above_ema200:
                        print(f"No entry: Close ${current_price:.2f} below EMA200 ${current_row['ema_200']:.2f}")
                    if not above_vwap:
                        print(f"No entry: Close ${current_price:.2f} below Session VWAP ${current_vwap:.2f}")

            # Brief pause between tickers
            await asyncio.sleep(0.5)

        # Wait 5 minutes before next scan
        print(f"\n{'='*60}")
        print(f"Scan complete. Waiting 5 minutes...")
        print(f"{'='*60}")
        await asyncio.sleep(300)


# Run the trading loop
try:
    ib.run(check_and_trade())
except Exception as e:
    print(f"An error occurred: {e}")
    import traceback
    traceback.print_exc()
finally:
    ib.disconnect()
    print("Disconnected from IB.")

Error 200, reqId 5: No security definition has been found for the request, contract: Stock(symbol='SAVA', exchange='SMART', currency='USD')
Error 200, reqId 31: No security definition has been found for the request, contract: Stock(symbol='BRK.B', exchange='SMART', currency='USD')
Unknown contract: Stock(symbol='SAVA', exchange='SMART', currency='USD')
Unknown contract: Stock(symbol='BRK.B', exchange='SMART', currency='USD')



Evaluating WMT at 08:33:51


Error 10349, reqId 75: Order TIF was set to GTC based on order preset.
Canceled order: Trade(contract=Stock(conId=13824, symbol='WMT', exchange='SMART', primaryExchange='NASDAQ', currency='USD', localSymbol='WMT', tradingClass='NMS'), order=MarketOrder(orderId=75, clientId=881, action='SELL', totalQuantity=20), orderStatus=OrderStatus(orderId=75, status='Cancelled', filled=0.0, remaining=0.0, avgFillPrice=0.0, permId=0, parentId=0, lastFillPrice=0.0, clientId=0, whyHeld='', mktCapPrice=0.0), fills=[], log=[TradeLogEntry(time=datetime.datetime(2026, 4, 1, 12, 33, 51, 869039, tzinfo=datetime.timezone.utc), status='PendingSubmit', message='', errorCode=0), TradeLogEntry(time=datetime.datetime(2026, 4, 1, 12, 33, 51, 873046, tzinfo=datetime.timezone.utc), status='Cancelled', message='Error 10349, reqId 75: Order TIF was set to GTC based on order preset.', errorCode=10349)], advancedError='')


Current Price:    $124.28
Trend Score:       -57.4/100
EMA 200:          $124.18
Session VWAP:     $124.38
Close > EMA200:    True
Close > VWAP:      False
Strong Bull Signal: False
Open Position - Entry: $122.72, P&L: 1.27%, Trend: -57.4
✅ TAKE PROFIT executed for WMT at $124.28 | P&L: $31.20

Evaluating MU at 08:33:51
Current Price:    $348.39
Trend Score:       49.3/100
EMA 200:          $337.02
Session VWAP:     $345.84
Close > EMA200:    True
Close > VWAP:      True
Strong Bull Signal: False
No entry: No strong bull signal (trend: 49.3)


Error 200, reqId 77: No security definition has been found for the request, contract: Stock(symbol='SAVA', exchange='SMART', currency='USD')


...

Current Price:    $95.23
Trend Score:       54.8/100
EMA 200:          $93.70
Session VWAP:     $95.27
Close > EMA200:    True
Close > VWAP:      False
Strong Bull Signal: False
Already traded UAL today. Skipping.

Evaluating BRK.B at 16:40:41
No historical data for BRK.B

Evaluating IBM at 16:40:41
Current Price:    $243.63
Trend Score:       -33.6/100
EMA 200:          $243.43
Session VWAP:     $243.78
Close > EMA200:    True
Close > VWAP:      False
Strong Bull Signal: False
Open Position - Entry: $245.18, P&L: -0.63%, Trend: -33.6

Evaluating MSFT at 16:40:42
Current Price:    $369.77
Trend Score:       -35.2/100
EMA 200:          $370.98
Session VWAP:     $370.69
Close > EMA200:    False
Close > VWAP:      False
Strong Bull Signal: False
No entry: No strong bull signal (trend: -35.2)
No entry: Close $369.77 below EMA200 $370.98
No entry: Close $369.77 below Session VWAP $370.69

Evaluating KO at 16:40:42
Current Price:    $76.08
Trend Score:       -55.6/100
EMA 200:          $76.0

Error 200, reqId 6114: No security definition has been found for the request, contract: Stock(symbol='SAVA', exchange='SMART', currency='USD')


Current Price:    $369.89
Trend Score:       45.7/100
EMA 200:          $356.58
Session VWAP:     $365.77
Close > EMA200:    True
Close > VWAP:      True
Strong Bull Signal: False
Already traded MU today. Skipping.

Evaluating SAVA at 16:46:10
No historical data for SAVA

Evaluating SLDB at 16:46:10
Current Price:    $7.38
Trend Score:       -17.7/100
EMA 200:          $7.26
Session VWAP:     $7.44
Close > EMA200:    True
Close > VWAP:      False
Strong Bull Signal: False
Already traded SLDB today. Skipping.

Evaluating SLV at 16:46:10
Current Price:    $68.03
Trend Score:       -55.5/100
EMA 200:          $67.94
Session VWAP:     $68.15
Close > EMA200:    True
Close > VWAP:      False
Strong Bull Signal: False
No entry: No strong bull signal (trend: -55.5)
No entry: Close $68.03 below Session VWAP $68.15

Evaluating NEM at 16:46:11
Current Price:    $113.79
Trend Score:       -55.1/100
EMA 200:          $112.10
Session VWAP:     $113.69
Close > EMA200:    True
Close > VWAP:      True


Error 200, reqId 6140: No security definition has been found for the request, contract: Stock(symbol='BRK.B', exchange='SMART', currency='USD')


Current Price:    $95.20
Trend Score:       55.4/100
EMA 200:          $93.72
Session VWAP:     $95.27
Close > EMA200:    True
Close > VWAP:      False
Strong Bull Signal: False
Already traded UAL today. Skipping.

Evaluating BRK.B at 16:46:26
No historical data for BRK.B

Evaluating IBM at 16:46:26
Current Price:    $243.63
Trend Score:       -30.2/100
EMA 200:          $243.43
Session VWAP:     $243.78
Close > EMA200:    True
Close > VWAP:      False
Strong Bull Signal: False
Open Position - Entry: $245.18, P&L: -0.63%, Trend: -30.2

Evaluating MSFT at 16:46:27
Current Price:    $369.69
Trend Score:       -31.7/100
EMA 200:          $370.96
Session VWAP:     $370.69
Close > EMA200:    False
Close > VWAP:      False
Strong Bull Signal: False
No entry: No strong bull signal (trend: -31.7)
No entry: Close $369.69 below EMA200 $370.96
No entry: Close $369.69 below Session VWAP $370.69

Evaluating KO at 16:46:28
Current Price:    $76.08
Trend Score:       -55.8/100
EMA 200:          $76.0